# 30_gdrive_playground.ipynb

Reading from the shared dataset. Writing a new datasheet with articles.

In [1]:
import gspread
from google.oauth2.service_account import Credentials

from dotenv import load_dotenv
import os

load_dotenv()  # Automatically finds .env file
url_sin_detenidos = os.getenv('GDRIVE_SIN_DETENIDOS')

import pandas as pd

## Example with valid header row

The first row must have unique values to be a valid header.

In [ ]:

# Define the scopes required
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

# Authenticate using the service account credentials JSON file
creds = Credentials.from_service_account_file("../secrets/credentials.json", scopes=SCOPES)
client = gspread.authorize(creds)

# Option A: Open sheet by URL (recommended)
sheet_url = "https://docs.google.com/spreadsheets/d/16oWJF6zePCwwAkxuETpFaS_7hRqfZROj8Jkgb_sxrto/edit"
spreadsheet = client.open_by_url(sheet_url)

# Option B: Open sheet by title
# spreadsheet = client.open("Exact Name of the Sheet")

# Select the first worksheet (tab)
worksheet = spreadsheet.get_worksheet(0)

# Extract data
all_records = worksheet.get_all_records()  # Returns data as a list of dictionaries
# all_values = worksheet.get_all_values()  # Alternative: Returns raw 2D list

# Example output
for row in all_records[:5]:  # Print first 5 rows
    print(row)

# {'ID': 1, 'ocurrio_en_CABA': 'SI', 'ubicacion': 'Villa 20 de Lugano (Chilavert y Araujo)', 'fuerza_de_seguridad': 'Policía de la Ciudad', 'fecha_del_hecho': '25 de diciembre'}
# {'ID': 2, 'ocurrio_en_CABA': 'SI', 'ubicacion': 'Parque Rivadavia, calles Rosario y Viel, barrio de Caballito, CABA', 'fuerza_de_seguridad': 'Policía Metropolitana', 'fecha_del_hecho': '7 de febrero de 2015'}
# {'ID': 3, 'ocurrio_en_CABA': 'SI', 'ubicacion': 'Congreso hacia Plaza de Mayo, Ciudad de Buenos Aires', 'fuerza_de_seguridad': 'Policía de la Ciudad y otras fuerzas provinciales/federales mencionadas en los casos', 'fecha_del_hecho': '26 de agosto de 2022 (fecha de la marcha)'}
# {'ID': 4, 'ocurrio_en_CABA': 'NO', 'ubicacion': 'Venezuela', 'fuerza_de_seguridad': 'Guardia de Honor Presidencial y Dirección de Contrainteligencia Militar (DGCIM)', 'fecha_del_hecho': 'NO SE PUEDE DETERMINAR'}
# {'ID': 5, 'ocurrio_en_CABA': 'NO SE PUEDE DETERMINAR', 'ubicacion': 'NO SE PUEDE DETERMINAR', 'fuerza_de_seguridad': 'NO SE PUEDE DETERMINAR', 'fecha_del_hecho': 'NO SE PUEDE DETERMINAR'}

{'ID': 1, 'ocurrio_en_CABA': 'SI', 'ubicacion': 'Villa 20 de Lugano (Chilavert y Araujo)', 'fuerza_de_seguridad': 'Policía de la Ciudad', 'fecha_del_hecho': '25 de diciembre'}
{'ID': 2, 'ocurrio_en_CABA': 'SI', 'ubicacion': 'Parque Rivadavia, calles Rosario y Viel, barrio de Caballito, CABA', 'fuerza_de_seguridad': 'Policía Metropolitana', 'fecha_del_hecho': '7 de febrero de 2015'}
{'ID': 3, 'ocurrio_en_CABA': 'SI', 'ubicacion': 'Congreso hacia Plaza de Mayo, Ciudad de Buenos Aires', 'fuerza_de_seguridad': 'Policía de la Ciudad y otras fuerzas provinciales/federales mencionadas en los casos', 'fecha_del_hecho': '26 de agosto de 2022 (fecha de la marcha)'}
{'ID': 4, 'ocurrio_en_CABA': 'NO', 'ubicacion': 'Venezuela', 'fuerza_de_seguridad': 'Guardia de Honor Presidencial y Dirección de Contrainteligencia Militar (DGCIM)', 'fecha_del_hecho': 'NO SE PUEDE DETERMINAR'}
{'ID': 5, 'ocurrio_en_CABA': 'NO SE PUEDE DETERMINAR', 'ubicacion': 'NO SE PUEDE DETERMINAR', 'fuerza_de_seguridad': 'NO SE 

In [3]:
# Convert records directly to DataFrame
df = pd.DataFrame(all_records)
df.head()

,ID,ocurrio_en_CABA,ubicacion,fuerza_de_seguridad,fecha_del_hecho
0,1,SI,Villa 20 de Lugano (Chilavert y Araujo),Policía de la Ciudad,25 de diciembre
1,2,SI,"Parque Rivadavia, calles Rosario y Viel, barri...",Policía Metropolitana,7 de febrero de 2015
2,3,SI,"Congreso hacia Plaza de Mayo, Ciudad de Buenos...",Policía de la Ciudad y otras fuerzas provincia...,26 de agosto de 2022 (fecha de la marcha)
3,4,NO,Venezuela,Guardia de Honor Presidencial y Dirección de C...,NO SE PUEDE DETERMINAR
4,5,NO SE PUEDE DETERMINAR,NO SE PUEDE DETERMINAR,NO SE PUEDE DETERMINAR,NO SE PUEDE DETERMINAR


## Sin Detenidos

Alternativa cuando el encabezado no está bien definido. Por ejemplo, si hay columnas con la primer fila vacía.

In [8]:
# Define the scopes required
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

# Authenticate using the service account credentials JSON file
creds = Credentials.from_service_account_file("../secrets/credentials.json", scopes=SCOPES)
client = gspread.authorize(creds)

spreadsheet = client.open_by_url(url_sin_detenidos)

worksheet = spreadsheet.get_worksheet(0)

# 1. Get raw cell values as a 2D list
raw_values = worksheet.get_all_values()

# 2. Define your own custom header names
headers = ['Fecha_deteccion', 'Medio', 'Titulo', 'Link', 'Keyword_detectada',
        'Fuente', 'Revisado', 'Validado', 'Observaciones', 'Estado_IA',
        'Palabras_detectadas', 'Puntaje', 'Puntaje_solo_titulo', 'otro_2']

# 3. Map values to records (skipping row 0 if row 0 has the old headers)
all_records = gspread.utils.to_records(headers, raw_values[1:])

# Example output
for row in all_records[:5]:  # Print first 5 rows
    print(row)

{'Fecha_deteccion': 'desde aca ejecuté el nuevo código', 'Medio': '', 'Titulo': '', 'Link': '', 'Keyword_detectada': '', 'Fuente': '', 'Revisado': 'FALSE', 'Validado': '', 'Observaciones': '', 'Estado_IA': '', 'Palabras_detectadas': '', 'Puntaje': '', 'Puntaje_solo_titulo': 'scores de títulos', 'otro_2': ''}
{'Fecha_deteccion': '4/8/2026', 'Medio': 'Google News', 'Titulo': 'Facundo Moyano fue demorado en Belgrano: pelea de pareja y estupefacientes - andigital.com.ar', 'Link': 'https://news.google.com/rss/articles/CBMisAFBVV95cUxOMW5QRFdDeFRrZkVKbGFqTHljUnVMQkVSV1pRazQ4eXo3ajAwR1kzMDRWN2JaY01HOEdpeEp0TmFvQS04Z2VEMDdXQ3AyT29lemwzRHhTUURndUhrRDAtWU0tT3R1N1pmejd6c2hfQ0I5UnV2emlQdmthczZUX0FIbkRpY2NEY0Z0c0k4ZkJUNFhWMWZxdHF3ZTNndVR2ZDRkdU5MSW1PYmItU1kwd0lwUw?oc=5', 'Keyword_detectada': 'demorado', 'Fuente': 'https://news.google.com/rss/search?q=demorado%20CABA&hl=es-419&gl=AR&ceid=AR:es-419', 'Revisado': 'FALSE', 'Validado': 'PENDIENTE', 'Observaciones': '', 'Estado_IA': 'PROCESADO', 'Palabra

In [9]:
# Convert records directly to DataFrame
df = pd.DataFrame(all_records)

# Remove invalid rows: no Title
col = df.columns[2]

# Filter out empty/whitespace strings and NaNs
df = df[df[col].astype(str).str.strip().ne("") & df[col].notna()]
df.head()

,Fecha_deteccion,Medio,Titulo,Link,Keyword_detectada,Fuente,Revisado,Validado,Observaciones,Estado_IA,Palabras_detectadas,Puntaje,Puntaje_solo_titulo,otro_2
1,4/8/2026,Google News,Facundo Moyano fue demorado en Belgrano: pelea...,https://news.google.com/rss/articles/CBMisAFBV...,demorado,https://news.google.com/rss/search?q=demorado%...,FALSE,PENDIENTE,,PROCESADO,"POLICIA: policia, efectivo, uniformado, comisa...",12,6,
2,4/8/2026,Google News,"Quién es Facundo Moyano, el dirigente sindical...",https://news.google.com/rss/articles/CBMixwFBV...,demorado,https://news.google.com/rss/search?q=demorado%...,FALSE,PENDIENTE,,PROCESADO,,,5,saltea filas consecutivas y excede tiempo de e...
3,4/8/2026,Google News,“Marcha de la gorra” | Miles de personas movil...,https://news.google.com/rss/articles/CBMirwFBV...,gatillo,https://news.google.com/rss/search?q=represion...,FALSE,PENDIENTE,,PROCESADO,,,3,
4,4/8/2026,Google News,Represión en Puente Pueyrredón: la UTEP denunc...,https://news.google.com/rss/articles/CBMizgFBV...,represión,https://news.google.com/rss/search?q=represion...,FALSE,PENDIENTE,,PROCESADO,"POLICIA: policia, efectivo, policial, DIR, fue...",13,6,
5,4/8/2026,Google News,"Corridas, represión y detenidos en el Obelisco...",https://news.google.com/rss/articles/CBMiXkFVX...,represión,https://news.google.com/rss/search?q=represion...,FALSE,PENDIENTE,,PROCESADO,,,9,


GNEWS:

Busco artículos en un periodo, usando las palabras clave de las categorías. Hago una búsqueda por cada término, de forma que un mismo artículo aparecerá varias veces. Luego calculo los puntajes igual al Apps Script, y además otros posibles puntajes.

In [ ]:
from gnews import GNews


In [89]:
excluded = ["gestion.pe"]

In [90]:
# gn = GNews()
gn = GNews(language="es-419", country="AR", exclude_websites=excluded)
# gn = GNews(language="es-419")

# gn.exclude_websites(excluded)

# gn.country = 'AR'  # News from a specific country 
# gn.language = 'es-419'  # News in a specific language
# gn.start_date = (2026, 8, 27)
# gn.end_date = (2026, 8, 28)

In [59]:
gn.start_date = (2018, 6, 18)
gn.end_date = (2018, 6, 20)

In [91]:
# articles = gn.get_news('gatillo')
articles = gn.get_news('gatillo after:2026-06-18 before:2026-06-20')
len(articles)

10

In [92]:
df = pd.DataFrame(articles)
df

,title,description,published date,url,publisher
0,Daredevil y Castigador: El gatillo del diablo ...,Daredevil y Castigador: El gatillo del diablo ...,"Sat, 20 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMif0FVX...,"{'href': 'https://www.zonanegativa.com', 'titl..."
1,CONTRA los DOCENTES: ZDERO vuelve a DEMORAR la...,CONTRA los DOCENTES: ZDERO vuelve a DEMORAR la...,"Thu, 18 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMi2AFBV...,"{'href': 'https://www.eldestapeweb.com', 'titl..."
2,Comenzó la paritaria docente - Tiempo Sur,Comenzó la paritaria docente Tiempo Sur,"Thu, 18 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMid0FVX...,"{'href': 'https://www.tiemposur.com.ar', 'titl..."
3,El principal negociador del régimen de Irán co...,El principal negociador del régimen de Irán co...,"Fri, 19 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMi9gFBV...,"{'href': 'https://www.infobae.com', 'title': '..."
4,'Amaro. Con el dedo en el gatillo': una mirada...,'Amaro. Con el dedo en el gatillo': una mirada...,"Fri, 19 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMijAFBV...,"{'href': 'https://www.milenio.com', 'title': '..."
5,Irán. Ejército iraní listo y bajo órdenes del ...,Irán. Ejército iraní listo y bajo órdenes del ...,"Fri, 19 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMixgFBV...,{'href': 'https://www.resumenlatinoamericano.o...
6,"Educación: Provincia ofreció un 16,8% de aumen...","Educación: Provincia ofreció un 16,8% de aumen...","Thu, 18 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMitgFBV...,"{'href': 'https://www.tiemposur.com.ar', 'titl..."
7,“Wilkyns Gatillo” archivos - Remolacha,“Wilkyns Gatillo” archivos Remolacha,"Thu, 18 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMiVEFVX...,"{'href': 'https://remolacha.net', 'title': 'Re..."
8,"Colas kilométricas en Aldi, a partir de hoy, p...","Colas kilométricas en Aldi, a partir de hoy, p...","Sat, 20 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMigAJBV...,"{'href': 'https://www.ultimahora.es', 'title':..."
9,DICRIM ocupa armas y más de 350 porciones de d...,DICRIM ocupa armas y más de 350 porciones de d...,"Thu, 18 Jun 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMi0gFBV...,"{'href': 'https://ensegundos.do', 'title': 'En..."


In [71]:
print(df.iloc[9]['url'])

https://news.google.com/rss/articles/CBMigAJBVV95cUxOd1lHMEtxVGJLYTBBcWI0eV83aF9DV1pSRUE3V2ZWdG10Q3NSZDJyVTh2QWZJSzJtb29Gak5IV1IyRHV5ZkRUck85WXRUbHhNUEdMUXRraGFpSTZua1VpN0Y5cDgwNWZITjh0NGRYS3VMTlA2MzdxOUdiTHpYQndCMkVJenI0cnJ2S0dOQ3RScmJhMWhTTDlNbkc3OUhoTk5yTGM0TnEtZklBcl9zTW01V0EzeERqaHEyb1JkLVA2UVMzNkF1WjJ4MUhNQTJPaXRzVXNIQ0g4WHVYZG54UmdTRVBUem13YUhmUXp4NkdPT1VMbVk4WThNbzhBRkNkR2dO0gGAAkFVX3lxTE5QOXloV0dQOEpHcktjMnFVLXpUeXI2LV9aZm52TDRmX0FER195RDk5WGFkRnd2RzhNZjBnbUd1RUp4Nldxbl9FdEFZbzNKb1hxZFdocHBidmt3SVBUaHptVERmZHRIZTFIQTZwVlcwcE13RHBPX1lhWmpoaGUxRWxJTDVRcUtTTldDNS16TF9nSi03ZDE1NlVlZDZDWGtzU1Z5Rm1iUG01bU50NTJpa3RKUkF4ajN2QUhJNGNTREIweHNKdHlNZWxEaG5lNzRqSktCTWN1dTlibmdNMmJlUUxoTUpXQjhHOFZfTVYyVWVPbUh4NDVjNTZOWGdLZnVJdGM?oc=5&hl=en-US&gl=US&ceid=US:en


In [77]:
gn = GNews()
gn.get_news_by_location("Pakistan")

[]

In [ ]:
df = pd.DataFrame(articles)

publisher_cols = pd.json_normalize(df['publisher']).set_index(df.index)
df = pd.concat([df.drop(columns=['publisher']), publisher_cols], axis=1)
df

,title,description,published date,url,href,title
0,Manifiesto volcánico por la 12° Marcha Contra ...,Manifiesto volcánico por la 12° Marcha Contra ...,"Thu, 27 Aug 2026 13:15:23 GMT",https://news.google.com/rss/articles/CBMioAFBV...,https://enfantterrible.com.ar,enfantterrible.com.ar
1,Interruptor De Gatillo SR-010 Para Amoladora A...,Interruptor De Gatillo SR-010 Para Amoladora A...,"Thu, 27 Aug 2026 22:08:29 GMT",https://news.google.com/rss/articles/CBMijgFBV...,https://insidegnss.com,Inside GNSS
2,Contra el Gatillo Fácil: familiares de víctima...,Contra el Gatillo Fácil: familiares de víctima...,"Fri, 28 Aug 2026 02:48:45 GMT",https://news.google.com/rss/articles/CBMivgFBV...,https://lmdiario.com.ar,La Nueva Mañana
3,“Manifestación volcánica”: la Marcha contra el...,“Manifestación volcánica”: la Marcha contra el...,"Thu, 27 Aug 2026 20:00:33 GMT",https://news.google.com/rss/articles/CBMisgFBV...,https://hoydia.com.ar,Hoy Día Córdoba
4,Hay 1.274 asesinados por el aparato represivo ...,Hay 1.274 asesinados por el aparato represivo ...,"Fri, 28 Aug 2026 00:27:48 GMT",https://news.google.com/rss/articles/CBMi3AFBV...,https://lavaca.org,lavaca.org
5,Marcha contra el Gatillo Fácil: familias vuelv...,Marcha contra el Gatillo Fácil: familias vuelv...,"Thu, 27 Aug 2026 18:51:01 GMT",https://news.google.com/rss/articles/CBMivAFBV...,https://lmdiario.com.ar,La Nueva Mañana
6,Judiciales de Entre Ríos contra el proyecto qu...,Judiciales de Entre Ríos contra el proyecto qu...,"Thu, 27 Aug 2026 14:07:31 GMT",https://news.google.com/rss/articles/CBMi1wFBV...,https://www.unoentrerios.com.ar,Uno Entre Rios
7,Pistola De Agua Pesada Vikan – Latón Azul Para...,Pistola De Agua Pesada Vikan – Latón Azul Para...,"Thu, 27 Aug 2026 14:02:55 GMT",https://news.google.com/rss/articles/CBMiYkFVX...,https://insidegnss.com,Inside GNSS
8,Reapareció la cazadora de animales - Agencia NOVA,Reapareció la cazadora de animales Agencia NOVA,"Thu, 27 Aug 2026 13:01:50 GMT",https://news.google.com/rss/articles/CBMifEFVX...,https://www.agencianova.com,Agencia NOVA
9,Tuerca De Liberación Rápida M14 Sin Llave Para...,Tuerca De Liberación Rápida M14 Sin Llave Para...,"Thu, 27 Aug 2026 19:49:52 GMT",https://news.google.com/rss/articles/CBMikgFBV...,https://insidegnss.com,Inside GNSS


In [ ]:
keywords = ['Travel & Tourism', 'Climate Change', 'Cryptocurrency']
all_news = []

for kw in keywords:
    news = gn.get_news(kw)
    for article in news:
        article['keyword'] = kw  # tag with keyword
    all_news.extend(news)

df_keywords = pd.DataFrame(all_news)
df_keywords.head()